# SDTM DM Domain Transformation

---
## Overview
Notebook 4 of 8.

This notebook transforms raw demographic data into SDTM Demographics (DM) domain format.

Run the entire notebook to load the data.

---

## Source Data

- **Source:** bronze.demog_raw
- **Domain:** Demographics (DM)

---

## Transformation Steps

**1. Data Quality**  
Loads bronze data and transforms records to domain standard

**2. SDTM Mapping**  
Maps source columns to SDTM variables:
- Subject identifiers (USUBJID, SUBJID)
- Demographics (AGE, SEX, RACE)
- Study metadata (STUDYID, SITEID, COUNTRY)
- Reference dates (RFSTDTC, BRTHDTC)

**3. Value Standardization**  
Applies controlled terminology:
- Sex codes (1=M, 2=F)
- Race decoding (Indicative)
- Date formatting (ISO 8601)

---

## Target

SDTM-compliant data written to **silver.sdtm_dm** Delta table.

---

In [0]:
from pyspark.sql.functions import udf, col, lit, when
from pyspark.sql.types import StringType

In [0]:
#global variables
CATALOG = "cluepoints_db_demo"
SCHEMA = "silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
# udf to decode sex
# note that Python UDFs are very slow on Spark as they get serialised/deserialised across the JVM boundary on every row
# to get 100x performance, write the UDF in Scala.
# I can't use Scala in this notebook as I'm using serverless compute which doesn't yet support Scala
# other option is to use Pandas vectorised UDF or write in-line Spark function

@udf(returnType=StringType())
def decode_sex(sex_code):
    if sex_code == 1:
        return 'M'
    elif sex_code == 2:
        return 'F'
    return None

In [0]:
#udf to decode race - indicative only, unsure of the real codes
@udf(returnType=StringType())
def decode_race(race_code):
    if race_code == 1:
        return 'RACE1'
    return None

In [0]:
#decode rand code as to whether someone receives placebo or not

@udf(returnType=StringType())
def decode_arm(rand_code):
    if rand_code == 0:
        return 'Placebo'
    elif rand_code == 1:
        return 'Treatment'
    return None

In [0]:
demog = spark.table("cluepoints_db_demo.bronze.demog")

#create spark dataframe - alias the columns and add the decode UDFs
#aliasing the columns forces the naming of the columns on the target table
sdtm_demog = demog.select(
    col('subjid').alias('SUBJID'),
    col('center').cast('string').alias('SITEID'),
    col('age_n').alias('AGE'),
    decode_sex(col('sex')).alias('SEX'),
    decode_race(col('race')).alias('RACE'),
    col('country').alias('COUNTRY'),
    col('birthdate').alias('BRTHDTC'),
    decode_arm(col("rand")).alias('ARM'),
    col('consda').alias('RFSTDTC'), #not totally sure about the mapping - study reference start datetime?
    col('source_file'),
    col('loaded_at'),
    col('ingestion_id')
)


In [0]:
# Write to Delta - overwrite all target data
sdtm_demog.write.format('delta').mode('overwrite').saveAsTable('cluepoints_db_demo.silver.demog_sdtm')

In [0]:
#display the table
display(spark.sql("select * from cluepoints_db_demo.silver.demog_sdtm limit 10"))

SUBJID,SITEID,AGE,SEX,RACE,COUNTRY,BRTHDTC,ARM,RFSTDTC,source_file,loaded_at,ingestion_id
840414004,414,35,M,RACE1,Italy,1969-09-10,Treatment,2005-05-31,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414002,414,59,F,RACE1,Italy,1945-09-30,Treatment,2005-04-29,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414006,414,70,M,RACE1,Italy,1935-12-31,Placebo,2005-06-03,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414001,414,58,M,RACE1,Italy,1946-09-30,Placebo,2005-04-28,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414003,414,77,M,RACE1,Italy,1928-08-10,Placebo,2005-05-26,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414005,414,64,F,RACE1,Italy,1940-09-05,Placebo,2005-06-03,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414007,414,64,M,RACE1,Italy,1941-02-11,Placebo,2005-07-05,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
840414099,414,99,M,RACE1,Italy,1927-02-11,Treatment,2004-02-05,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
826290006,290,85,M,RACE1,Spain,1920-12-31,Treatment,2005-06-09,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
826290002,290,78,F,RACE1,Spain,1925-10-25,Treatment,2004-08-10,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
